4) 대화 엔티티 메모리

In [ ]:
# [목적] ConversationEntityMemory가 사용하는 기본 프롬프트 구조를 확인하는 예제
# 엔터티 메모리와 관련 템플릿을 불러온 뒤, 대화에서 인물·장소·조직 정보를 추출하는 프롬프트를 출력합니다.
# 이후 체인이 어떤 규칙으로 중요한 대화 대상을 기억하는지 이해하기 위해 먼저 확인합니다.
from langchain_openai import ChatOpenAI
from langchain_classic.chains import ConversationChain
from langchain_classic.memory import ConversationEntityMemory
from langchain_classic.memory.prompt import ENTITY_MEMORY_CONVERSATION_TEMPLATE
from dotenv import load_dotenv

load_dotenv()

# ENTITY_MEMORY_CONVERSATION_TEMPLATE은 엔터티 추출과 요약에 쓰이는 LangChain 기본 프롬프트입니다.
print(ENTITY_MEMORY_CONVERSATION_TEMPLATE.template)

In [ ]:
# [목적] 엔터티 메모리를 연결한 대화 체인을 구성하는 예제
# 채팅 모델, 엔터티 전용 메모리, 기본 프롬프트를 하나의 ConversationChain으로 묶습니다.
# 이렇게 만든 체인은 대화 중 등장한 주요 대상의 정보를 별도로 기억하며 응답합니다.
# temperature=0은 같은 입력에 대해 답변의 변화를 줄여 일관된 결과를 얻도록 합니다.
llm = ChatOpenAI(model_name="gpt-4o", temperature=0)

conversation = ConversationChain(
    llm=llm,
    prompt=ENTITY_MEMORY_CONVERSATION_TEMPLATE,
    # ConversationEntityMemory는 인물, 회사처럼 대화에서 언급된 대상을 요약해 저장합니다.
    memory=ConversationEntityMemory(llm=llm),
)

In [ ]:
# [목적] 대화 입력을 처리하면서 등장한 엔터티 정보를 메모리에 저장하는 예제
# predict에 문장을 전달하면 체인이 모델의 응답을 만들고, 함께 언급된 사람과 조직의 정보를 추출합니다.
# 추출된 정보는 다음 셀에서 엔터티 저장소를 통해 확인합니다.
conversation.predict(
    input="테디와 설리는 한 회사에서 일하는 동료입니다."
    "테디는 개발자이고 설리는 디자이너입니다."
    "그들은 최근 회사에서 일하는 것을 그만두고 자신들의 회사를 차릴 계획을 세우고 있습니다."
)

In [ ]:
# [목적] 엔터티 메모리가 저장한 대상별 요약 정보를 확인하는 예제
# entity_store.store는 이름을 키로 하고 해당 대상의 대화 요약을 값으로 보관하는 저장소입니다.
# 체인이 이후 대화에서 어떤 인물이나 조직 정보를 참조할 수 있는지 점검합니다.
conversation.memory.entity_store.store